In [ ]:
"""Assignment 2: Regression and Classification
Dataset: Kaggle "Student Performance Factors"
"""
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

SEED = 42
CSV = "StudentPerformanceFactors.csv"
sns.set_theme(style="whitegrid")


# ------------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------------
def make_synthetic(n=600, seed=SEED):
    rng = np.random.default_rng(seed)
    hours = rng.normal(20, 6, n).clip(1, 40).round()
    attend = rng.normal(80, 10, n).clip(55, 100).round()
    sleep = rng.normal(7, 1.2, n).clip(4, 10).round()
    prev = rng.normal(75, 14, n).clip(50, 100).round()
    tutor = rng.integers(0, 6, n)
    activity = rng.integers(0, 6, n)
    # diminishing returns on study hours -> mild curvature
    score = (5 + 0.35 * prev + 0.20 * attend + 1.0 * hours - 0.012 * hours ** 2
             + 0.5 * sleep + 0.4 * tutor + rng.normal(0, 3.0, n))
    return pd.DataFrame({
        "Hours_Studied": hours, "Attendance": attend, "Sleep_Hours": sleep,
        "Previous_Scores": prev, "Tutoring_Sessions": tutor,
        "Physical_Activity": activity, "Exam_Score": score.clip(0, 100).round(),
    })


if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    print(f"Loaded real Kaggle file: {CSV}")
else:
    df = make_synthetic()
    print("Kaggle CSV not found -> using synthetic data with the same columns")

cols = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores",
        "Tutoring_Sessions", "Physical_Activity", "Exam_Score"]
df = df[cols].dropna().drop_duplicates().reset_index(drop=True)
print("Shape:", df.shape)
print(df.describe().round(2), "\n")

FEATURES = ["Hours_Studied", "Attendance", "Previous_Scores", "Sleep_Hours"]  # 4 predictors
X, y = df[FEATURES], df["Exam_Score"]
idx_train, idx_test = train_test_split(df.index, test_size=0.25, random_state=SEED)
Xtr, Xte, ytr, yte = X.loc[idx_train], X.loc[idx_test], y.loc[idx_train], y.loc[idx_test]


def reg_metrics(name, y_true, y_pred):
    return {"Model": name,
            "R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))}


# ------------------------------------------------------------------
# Q1. Multivariate Linear Regression
# ------------------------------------------------------------------
lin = LinearRegression().fit(Xtr, ytr)
pred_lin = lin.predict(Xte)
print("=" * 60, "\nQ1. Multivariate Linear Regression")
print("Intercept:", round(lin.intercept_, 3))
for f, c in zip(FEATURES, lin.coef_):
    print(f"  {f:<18} coef = {c:+.3f}")
res_lin = reg_metrics("Linear (degree 1)", yte, pred_lin)
print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in res_lin.items()}, "\n")

# ------------------------------------------------------------------
# Q2. Polynomial Regression vs Linear Regression
# ------------------------------------------------------------------
print("=" * 60, "\nQ2. Polynomial Regression")
rows, preds_by_deg = [], {}
for d in [1, 2, 3, 4]:
    model = make_pipeline(PolynomialFeatures(d, include_bias=False),
                          StandardScaler(), LinearRegression())
    model.fit(Xtr, ytr)
    ptr, pte = model.predict(Xtr), model.predict(Xte)
    cv = cross_val_score(model, Xtr, ytr, cv=5, scoring="r2").mean()
    preds_by_deg[d] = pte
    rows.append({"Degree": d, "Train R2": r2_score(ytr, ptr), "Test R2": r2_score(yte, pte),
                 "CV R2 (5-fold)": cv, "Test MAE": mean_absolute_error(yte, pte),
                 "Test RMSE": np.sqrt(mean_squared_error(yte, pte))})
poly_table = pd.DataFrame(rows).round(4)
print(poly_table.to_string(index=False))
pred_poly = preds_by_deg[2]  # degree 2 used for the comparison below
res_poly = reg_metrics("Polynomial (degree 2)", yte, pred_poly)
print("\nLinear vs Polynomial(2) on the test set:")
print(pd.DataFrame([res_lin, res_poly]).round(3).to_string(index=False), "\n")

# ------------------------------------------------------------------
# Q3. Logistic Regression: Pass / Fail  (Pass if Exam_Score >= 65)
# ------------------------------------------------------------------
PASS_MARK = 65
df["Result"] = (df["Exam_Score"] >= PASS_MARK).astype(int)  # 1 = Pass, 0 = Fail
ybin_tr, ybin_te = df.loc[idx_train, "Result"], df.loc[idx_test, "Result"]
print("=" * 60, f"\nQ3. Binary Logistic Regression (Pass if score >= {PASS_MARK})")
print("Class counts:\n", df["Result"].map({1: "Pass", 0: "Fail"}).value_counts().to_string())
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logit.fit(Xtr, ybin_tr)
pred_bin = logit.predict(Xte)
print(f"\nAccuracy: {accuracy_score(ybin_te, pred_bin):.3f}")
print(classification_report(ybin_te, pred_bin, target_names=["Fail", "Pass"], digits=3))

# ------------------------------------------------------------------
# Q4. Multiclass Logistic Regression: Low / Medium / High
# ------------------------------------------------------------------
def band(s):
    return "Low" if s < 60 else ("Medium" if s < 70 else "High")

df["Performance"] = df["Exam_Score"].apply(band)
order = ["Low", "Medium", "High"]
ymc_tr, ymc_te = df.loc[idx_train, "Performance"], df.loc[idx_test, "Performance"]
print("=" * 60, "\nQ4. Multiclass Logistic Regression (Low <60, Medium 60-69, High >=70)")
print("Class counts:\n", df["Performance"].value_counts().reindex(order).to_string())
mlogit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
mlogit.fit(Xtr, ymc_tr)
pred_mc = mlogit.predict(Xte)
print(f"\nAccuracy: {accuracy_score(ymc_te, pred_mc):.3f}")
print(classification_report(ymc_te, pred_mc, labels=order, digits=3))

# ------------------------------------------------------------------
# Q5. Compare predictions with actual values
# ------------------------------------------------------------------
print("=" * 60, "\nQ5. Predictions vs actual (first 15 test students)")
cmp = pd.DataFrame({
    "Actual_Score": yte.values,
    "Linear_Pred": pred_lin.round(1),
    "Poly2_Pred": pred_poly.round(1),
    "Actual_Result": ybin_te.map({1: "Pass", 0: "Fail"}).values,
    "Pred_Result": pd.Series(pred_bin).map({1: "Pass", 0: "Fail"}).values,
    "Actual_Band": ymc_te.values,
    "Pred_Band": pred_mc,
})
print(cmp.head(15).to_string(index=False))
cmp.to_csv("predictions_vs_actual.csv", index=False)

# --- Figure 1: actual vs predicted for both regressions ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
for ax, p, nm, r in [(axes[0], pred_lin, "Linear Regression", res_lin),
                     (axes[1], pred_poly, "Polynomial Regression (deg 2)", res_poly)]:
    sns.scatterplot(x=yte, y=p, ax=ax, alpha=0.7, color="steelblue")
    lim = [min(yte.min(), p.min()) - 1, max(yte.max(), p.max()) + 1]
    ax.plot(lim, lim, "r--", label="Perfect prediction")
    ax.set_title(f"{nm}\nR²={r['R2']:.3f}, RMSE={r['RMSE']:.2f}")
    ax.set_xlabel("Actual Exam Score")
    ax.set_ylabel("Predicted Exam Score")
    ax.legend()
plt.tight_layout()
plt.savefig("fig1_actual_vs_predicted_regression.png", dpi=150)
plt.close()

# --- Figure 2: confusion matrices for the two classifiers ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
ConfusionMatrixDisplay(confusion_matrix(ybin_te, pred_bin),
                       display_labels=["Fail", "Pass"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Binary (Pass/Fail), acc={accuracy_score(ybin_te, pred_bin):.2f}")
ConfusionMatrixDisplay(confusion_matrix(ymc_te, pred_mc, labels=order),
                       display_labels=order).plot(ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title(f"Multiclass (Low/Medium/High), acc={accuracy_score(ymc_te, pred_mc):.2f}")
plt.tight_layout()
plt.savefig("fig2_confusion_matrices.png", dpi=150)
plt.close()

# --- Figure 3: polynomial degree vs R2 (shows under/overfitting) ---
plt.figure(figsize=(7, 4.5))
plt.plot(poly_table["Degree"], poly_table["Train R2"], "o-", label="Train R²")
plt.plot(poly_table["Degree"], poly_table["Test R2"], "s-", label="Test R²")
plt.xticks([1, 2, 3, 4])
plt.xlabel("Polynomial degree")
plt.ylabel("R²")
plt.title("Effect of polynomial degree")
plt.legend()
plt.tight_layout()
plt.savefig("fig3_poly_degree_vs_r2.png", dpi=150)
plt.close()

df.to_csv("students_with_labels.csv", index=False)
print("\nSaved: fig1..fig3 PNGs, predictions_vs_actual.csv, students_with_labels.csv")
